In [12]:

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found")

genai.configure(api_key=GEMINI_API_KEY)

# Single Gemini instance
gemini_model = genai.GenerativeModel("gemini-3-flash-preview")

# ------------------------------
# 2️⃣ Short-Term Memory (STM)
# ------------------------------
short_term_memory = deque(maxlen=20)  # store last 20 interactions

def add_to_short_memory(question, answer):
    short_term_memory.append({"question": question, "answer": answer})

def get_recent_memory():
    return list(short_term_memory)

# ------------------------------
# 3️⃣ Long-Term Memory (LTM) – FAISS
# ------------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
dimension = 384
index = faiss.IndexFlatL2(dimension)
memory_texts = []

def add_to_long_term_memory(text):
    """Store useful knowledge in FAISS index"""
    if len(text) > 50:  # only store meaningful info
        emb = embedding_model.encode([text])
        memory_texts.append(text)
        index.add(np.array(emb, dtype="float32"))

def query_long_term_memory(query, top_k=3):
    if len(memory_texts) == 0:
        return []
    query_emb = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_emb, dtype="float32"), top_k)
    return [memory_texts[i] for i in indices[0]]

# ------------------------------
# 4️⃣ Load Notebook 01 RAG output
# ------------------------------
def load_rag_answer(filename="responses_log.json"):
    """Load RAG output from Notebook 01"""
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("result", {})

def inject_rag_into_memory():
    """Add Notebook 01 answers into STM and LTM"""
    result = load_rag_answer()
    if not result:
        return

    question = result.get("question", "")
    answer = result.get("answer", "")
    if question and answer:
        add_to_short_memory(question, answer)
        add_to_long_term_memory(f"Q: {question}\nA: {answer}")

# Inject RAG data at notebook start
inject_rag_into_memory()

# ------------------------------
# 5️⃣ Build Prompt with Memory + RAG
# ------------------------------
def build_prompt(user_question, include_long_term=True):
    # Short-term context
    recent_mem = "\n".join([f"Q: {m['question']}\nA: {m['answer']}" for m in get_recent_memory()])

    # Long-term context
    long_term_context = ""
    if include_long_term:
        retrieved = query_long_term_memory(user_question)
        if retrieved:
            long_term_context = "\n".join([f"Relevant past knowledge:\n{r}" for r in retrieved])

    prompt = f"""
You are an AI assistant with memory.

Do not explicitly mention memory unless asked.

Recent conversation:
{recent_mem}

{long_term_context}

Current question:
{user_question}

Answer clearly, using RAG info if available.
"""
    return prompt.strip()

# ------------------------------
# 6️⃣ Multi-Turn Q&A Function
# ------------------------------
def multi_turn_ask(question):
    """Ask Gemini a question with STM + LTM context"""
    prompt = build_prompt(question)
    response = gemini_model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.3,
              "max_output_tokens": 200  
        }
    )
    answer = response.text.strip()

    # Store in STM
    add_to_short_memory(question, answer)

    # Store in LTM
    add_to_long_term_memory(f"Q: {question}\nA: {answer}")

    return answer

# ------------------------------
# 7️⃣ Test Multi-Turn Conversation
# ------------------------------
q1 = "Explain LiDAR in autonomous vehicles"
print("Gemini:", multi_turn_ask(q1))

q2 = "How is LiDAR different from radar?"
print("Gemini:", multi_turn_ask(q2))

q3 = "What sensing technologies did we already discuss?"
print("Gemini:", multi_turn_ask(q3))

q4 = "How do self-driving cars detect obstacles and pedestrians?"
print("Gemini:", multi_turn_ask(q4))


Gemini: LiDAR, which stands for **Light
Gemini: LiDAR and RADAR are both
Gemini: We have discussed the following sensing technologies used in self-driving
Gemini: Self-driving cars detect obstacles and pedestrians
